[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> 3. Help Assistant Segmented By Puzzle](https://colab.research.google.com/github/AlbertoLopezCorbalan/MIA-TFM/blob/main/TFM_help_assistant_25_segmented_by_puzzle.ipynb)  

# Trabajo Fin de Máster - Help Assistant Segmented By Puzzle
> Autor: Alberto López Corbalán alberto.lopezc@um.es

Con el objetivo de analizar el impacto de distintas estrategias de entrenamiento sobre el rendimiento del modelo, se llevaron a cabo dos enfoques experimentales diferenciados. En el primero, se entrenó un modelo global utilizando conjuntamente los datos de todos los puzzles disponibles, evaluando posteriormente su capacidad de predicción de manera individual para cada tarea. En el segundo enfoque, se aplicó un proceso de fine-tuning específico para cada puzzle, ajustando el modelo únicamente con ejemplos pertenecientes a esa tarea concreta. Para comparar ambos métodos, se utilizaron las métricas de precisión (*precision*), exhaustividad (*recall*) y puntuación F1 (*F1-score*).

Para esta problemática se requiere un entorno con GPU para poder ejecutarse.

In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Google Colab")
    # Para obtener el dataset si se ejecuta en Colab
    !git clone https://github.com/AlbertoLopezCorbalan/MIA-TFM
    %cd MIA-TFM

In [ ]:
import pandas as pd
import re
import numpy as np
import json
import matplotlib.pyplot as plt
import time
import torch
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV

# https://huggingface.co/allenai/longformer-base-4096 -> https://arxiv.org/pdf/2004.05150
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification

print(torch.cuda.get_device_name(0))

GLOBAL_SEED = 123 # Seed global utilizada para garantizar la reproducibilidad
DATASET_PATH = "dataset/complete_report_features_and_text_replay.csv"

Tesla T4


Para este problema solo será necesaria la traza, el puzzle y variable objetivo del dataset.


In [ ]:
dataset = pd.read_csv(DATASET_PATH)
print(f"{DATASET_PATH}: " + str(len(dataset)))
dataset.head()

dataset/complete_report_features_and_text_replay.csv: 6995


,replay,user,group,puzzle,globalAttemptId,attemptId,contextFeatures,attemptFeatures,textReplay
0,2. Separated Boxes~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,2. Separated Boxes,2,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 2, ""ActiveTi...","{""ActiveTime"": 27.355294999999998, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
1,3. Rotate a Pyramid~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,3. Rotate a Pyramid,3,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 3, ""ActiveTi...","{""ActiveTime"": 164.38663100000002, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
2,3. Rotate a Pyramid~2,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,3. Rotate a Pyramid,5,2,"{""#Attempt"": 2, ""#GlobalAttempt"": 5, ""ActiveTi...","{""ActiveTime"": 170.76343800000006, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
3,Bear Market~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,Bear Market,4,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 4, ""ActiveTi...","{""ActiveTime"": 40.556214, ""InactiveTime"": 0, ""...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
4,4. Match Silhouettes~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,4. Match Silhouettes,6,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 6, ""ActiveTi...","{""ActiveTime"": 100.48931000000002, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...


# Preprocesamiento 25%

A continuación, preparamos los datos acordes al experimento. Primero, parseamos los JSON contenidos en la columna attemptFeatures para extraer la variable objetivo `attempt_Completed`. Luego, seleccionamos únicamente la columna de texto `textReplay` junto con la variable objetivo. Finalmente, generamos la versión del DataFrame con un 25% del texto para entrenar y evaluar el modelo en distintos puzzles y evaluar su rendimiento, ya que en experimentaciones previas se observó que es en este segmento donde se concentran los patrones más relevantes para la predicción y donde el modelo obtiene los mejores resultados.

In [ ]:
df_llm = dataset.copy(deep = True)

# Parseamos el json
df_llm["attemptFeatures"] = df_llm["attemptFeatures"].apply(json.loads)

# Normalizamos los json para pasarlos a una tabla aplanada
df_attempt = pd.json_normalize(df_llm["attemptFeatures"])

# Quitamos los "." por "_"
df_attempt.columns = df_attempt.columns.str.replace(".", "_", regex=False)

# Unimos todo
df_llm = pd.concat(
    [
        df_llm.drop(columns=["contextFeatures", "globalAttemptId", "attemptFeatures"]),
        df_attempt.add_prefix("attempt_"),
    ],
    axis=1
)

# Seleccionamos solo la textReplay y la variable objetivo
df_llm = df_llm[["textReplay", "attempt_Completed", "puzzle"]]
df_llm["attempt_Completed"] = df_llm["attempt_Completed"].astype("int64")

# Omitimos los prefijos "1. ", "7. " para limpiar la cadena de caracteres
df_llm["puzzle"] = df_llm["puzzle"].apply(lambda x: re.sub(r"^\d+\.\s*", "", x).strip())

# DataFrame al 25%
df_llm_25 = df_llm.copy(deep=True)
df_llm_25["textReplay"] = df_llm_25["textReplay"].apply(lambda x: x[:int(len(x) * 0.25)])

# Entrenamiento

In [ ]:
def train_llm_classifier(df, max_length=4096):

  # Parámetros de configuración
  model_name = "allenai/longformer-base-4096"
  epochs = 1
  train_batch_size = 2
  eval_batch_size = 2
  learning_rate = 2e-5
  weight_decay = 0.01


  # Divide el dataset en entrenamiento y test
  # usando estratificación para mantener proporción de clases
  x_train, x_test, y_train, y_test = train_test_split(df["textReplay"].tolist(), df["attempt_Completed"].tolist(),
                                                      test_size=0.2, random_state=GLOBAL_SEED, stratify=df["attempt_Completed"])

  # Carga el tokenizer del modelo
  tokenizer = LongformerTokenizerFast.from_pretrained(model_name)

  # Se tokeniza del conjunto de entrenamiento y prueba
  train_encodings = tokenizer(x_train, truncation=True, padding=True, max_length=max_length)
  test_encodings = tokenizer(x_test, truncation=True, padding=True, max_length=max_length)

  # Se convierte a formato Dataset de HuggingFace
  train_dataset = Dataset.from_dict({**train_encodings, "labels": y_train})
  test_dataset = Dataset.from_dict({**test_encodings, "labels": y_test})

  # Carga del modelo para clasificación
  model = LongformerForSequenceClassification.from_pretrained(model_name, num_labels=2)

  # Función para calcular rendimiento
  def compute_metrics(eval_pred):

      logits, labels = eval_pred

      # Obtiene la clase con mayor probabilidad
      predictions = np.argmax(logits, axis=-1)

      return {
          "f1": f1_score(labels, predictions),
          "precision": precision_score(labels, predictions),
          "recall": recall_score(labels, predictions),
      }

  # Configuración del entrenamiento
  training_args = TrainingArguments(
      eval_strategy = "epoch",
      logging_strategy = "epoch",
      num_train_epochs = epochs,
      # En estas pruebas no guardaremos el mejor modelo
      load_best_model_at_end = False,
      save_strategy = "no",
      # Batch sizes
      per_device_train_batch_size = train_batch_size,
      per_device_eval_batch_size = eval_batch_size,
      learning_rate = learning_rate,
      weight_decay = weight_decay,
      fp16 = torch.cuda.is_available()
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics,
  )

  trainer.train()

  # Mostramos el rendimiento
  metrics = trainer.evaluate()

  return metrics

## Entrenamiento específico por puzzle

En este enfoque, se realizó un proceso de fine-tuning independiente para cada puzzle, utilizando exclusivamente los ejemplos pertenecientes a dicha tarea. El objetivo es analizar la capacidad del modelo para especializarse en patrones concretos y evaluar si un entrenamiento centrado únicamente en un puzzle concreto permite mejorar el rendimiento de predicción. Posteriormente, cada modelo ajustado se evalúa sobre los datos correspondientes a su propio puzzle utilizando las métricas de precision, recall y F1-score.

In [ ]:
unique_puzzles = df_llm_25['puzzle'].unique()
puzzle_metrics = {}

print(f"{len(unique_puzzles)} unique puzzles.")

for puzzle_name in unique_puzzles:
    print(f"\n--- Training for puzzle: {puzzle_name} ---")
    df_puzzle = df_llm_25[df_llm_25['puzzle'] == puzzle_name].copy(deep = True)

    total_examples = len(df_puzzle)
    completed_examples = df_puzzle['attempt_Completed'].sum()
    not_completed_examples = total_examples - completed_examples
    print(f"  Total examples for this puzzle: {total_examples}")
    print(f"  Examples with 'attempt_Completed' (True): {completed_examples}")
    print(f"  Examples without 'attempt_Completed' (False): {not_completed_examples}")


    metrics = train_llm_classifier(df_puzzle)
    puzzle_metrics[puzzle_name] = metrics


print("\n\n\n\n\n\n--- Summary of Metrics by Puzzle ---")
for puzzle, metrics in puzzle_metrics.items():
    print(f"Puzzle: {puzzle}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

31 unique puzzles.

--- Training for puzzle: Separated Boxes ---
  Total examples for this puzzle: 355
  Examples with 'attempt_Completed' (True): 310
  Examples without 'attempt_Completed' (False): 45


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


--- Training for puzzle: Tetromino ---
  Total examples for this puzzle: 12
  Examples with 'attempt_Completed' (True): 10
  Examples without 'attempt_Completed' (False): 2


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.dense.weight           | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight         | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
classifier.out_proj.weight     | MISSING    | 
classifier.dense.bias          | MISSING    | 
classifier.dense.weight        | MISSING    | 
classifier.out_proj.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.565308,0.617025,0.800000,0.666667,1.000000








--- Summary of Metrics by Puzzle ---
Puzzle: Separated Boxes
  eval_loss: 0.6684
  eval_f1: 0.9323
  eval_precision: 0.8732
  eval_recall: 1.0000
  eval_runtime: 10.0664
  eval_samples_per_second: 7.0530
  eval_steps_per_second: 3.5760
  epoch: 1.0000
Puzzle: Rotate a Pyramid
  eval_loss: 0.4478
  eval_f1: 0.9457
  eval_precision: 0.8971
  eval_recall: 1.0000
  eval_runtime: 36.2273
  eval_samples_per_second: 1.8770
  eval_steps_per_second: 0.9390
  epoch: 1.0000
Puzzle: Bear Market
  eval_loss: 0.1709
  eval_f1: 0.0000
  eval_precision: 0.0000
  eval_recall: 0.0000
  eval_runtime: 10.5429
  eval_samples_per_second: 3.6990
  eval_steps_per_second: 1.8970
  epoch: 1.0000
Puzzle: Match Silhouettes
  eval_loss: 0.6855
  eval_f1: 0.8976
  eval_precision: 0.8143
  eval_recall: 1.0000
  eval_runtime: 14.4728
  eval_samples_per_second: 4.8370
  eval_steps_per_second: 2.4180
  epoch: 1.0000
Puzzle: One Box
  eval_loss: 0.4677
  eval_f1: 0.9583
  eval_precision: 0.9200
  eval_recall: 1.00

## Entrenamiento global

En contraste, el entrenamiento global se llevó a cabo utilizando conjuntamente los datos de todos los puzzles disponibles para entrenar un único modelo generalista. Este enfoque busca que el modelo aprenda representaciones y patrones compartidos entre distintas tareas, favoreciendo una mayor capacidad de generalización y estabilidad. Una vez finalizado el entrenamiento, el modelo global se evalúa individualmente sobre cada puzzle para comparar su comportamiento frente al enfoque especializado.

In [ ]:
# Parameters from the existing train_llm_classifier function
model_name = "allenai/longformer-base-4096"
epochs = 1
train_batch_size = 2
eval_batch_size = 2
learning_rate = 2e-5
weight_decay = 0.01
max_length = 4096

# Divide el dataset en entrenamiento y test
# usando estratificación para mantener proporción de clases
train_df, test_df = train_test_split(df_llm_25,
                                     test_size=0.2,
                                     random_state=GLOBAL_SEED,
                                     stratify=df_llm_25["attempt_Completed"])

x_train_global = train_df["textReplay"].tolist()
y_train_global = train_df["attempt_Completed"].tolist()

x_test_global = test_df["textReplay"].tolist()
y_test_global = test_df["attempt_Completed"].tolist()

# Se carga el tokenizer
global_tokenizer = LongformerTokenizerFast.from_pretrained(model_name)

# Se tokeniza del conjunto de entrenamiento y prueba
train_encodings_global = global_tokenizer(x_train_global, truncation=True, padding=True, max_length=max_length)
test_encodings_global = global_tokenizer(x_test_global, truncation=True, padding=True, max_length=max_length)

# Se convierte a formato Dataset de HuggingFace
train_dataset_global = Dataset.from_dict({**train_encodings_global, "labels": y_train_global})
test_dataset_global = Dataset.from_dict({**test_encodings_global, "labels": y_test_global})

# Carga del modelo para clasificación
global_model = LongformerForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Función para calcular rendimiento
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1) # Obtiene la clase con mayor probabilidad
    return {
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
    }

# Configuración del entrenamiento
training_args_global = TrainingArguments(
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy = "epoch",
    num_train_epochs=epochs,
    load_best_model_at_end=True,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=torch.cuda.is_available()
)


global_trainer = Trainer(
    model=global_model,
    args=training_args_global,
    train_dataset=train_dataset_global,
    eval_dataset=test_dataset_global,
    compute_metrics=compute_metrics,
)

global_trainer.train()

# Mostramos el rendimiento global
global_eval_metrics = global_trainer.evaluate()
print(f"Global Evaluation Metrics: {global_eval_metrics}")

# Evaluación por puzzle
print("\n\n--- Evaluating global model performance per puzzle ---")

unique_puzzles = test_df['puzzle'].unique()
puzzle_metrics_global_model = {}

for puzzle_name in unique_puzzles:
    print(f"\n--- Evaluating for puzzle: {puzzle_name} ---")

    puzzle_test_df = test_df[test_df['puzzle'] == puzzle_name]

    x_puzzle_test = puzzle_test_df["textReplay"].tolist()
    y_puzzle_test = puzzle_test_df["attempt_Completed"].tolist()

    puzzle_encodings = global_tokenizer(x_puzzle_test, truncation=True, padding=True, max_length=max_length)
    puzzle_dataset = Dataset.from_dict({**puzzle_encodings, "labels": y_puzzle_test})

    # Usamos el modelo global para predecir
    predictions_output = global_trainer.predict(puzzle_dataset)
    logits = predictions_output.predictions
    predicted_labels = np.argmax(logits, axis=-1)
    true_labels = predictions_output.label_ids

    # Calculamos las métricas
    metrics = {
        "f1": f1_score(true_labels, predicted_labels),
        "precision": precision_score(true_labels, predicted_labels),
        "recall": recall_score(true_labels, predicted_labels),
    }
    puzzle_metrics_global_model[puzzle_name] = metrics

print("\n\n\n\n\n\n--- Summary of Global Model Metrics by Puzzle ---")
for puzzle, metrics in puzzle_metrics_global_model.items():
    print(f"Puzzle: {puzzle}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...


--- Evaluating for puzzle: Few Clues ---



--- Evaluating for puzzle: Orange Dance ---



--- Evaluating for puzzle: Ramp Up and Can It ---



--- Evaluating for puzzle: Tetromino ---








--- Summary of Global Model Metrics by Puzzle ---
Puzzle: Object Limits
  f1: 0.7059
  precision: 0.5455
  recall: 1.0000
Puzzle: Boxes Obscure Spheres
  f1: 0.7123
  precision: 0.5532
  recall: 1.0000
Puzzle: Stranger Shapes
  f1: 0.5500
  precision: 0.3929
  recall: 0.9167
Puzzle: Separated Boxes
  f1: 0.9375
  precision: 0.8824
  recall: 1.0000
Puzzle: Zzz
  f1: 0.7647
  precision: 0.6500
  recall: 0.9286
Puzzle: Match Silhouettes
  f1: 0.9206
  precision: 0.8529
  recall: 1.0000
Puzzle: Square Cross-Sections
  f1: 0.7000
  precision: 0.5385
  recall: 1.0000
Puzzle: Not Bird
  f1: 0.6829
  precision: 0.5185
  recall: 1.0000
Puzzle: More Than Meets Your Eye
  f1: 0.6667
  precision: 0.5238
  recall: 0.9167
Puzzle: Combine 2 Ramps
  f1: 0.8960
  precision: 0.8116
  recall: 1.0000
Puzzle: Warm Up
  f1: 0.9259
  precision: 0.8621
  recall: 1.0000
Puzzle: Pyramids are Strange
  f1: 0.8421
  precision: 0.7273
  recall: 1.0000
Puzzle: Stretch a Ramp
  f1: 0.8496
  precision: 0.7385
 

# Conclusiones
| Puzzle | F1 Global (%) | Precisión Global (%) | Recall Global (%) | F1 Específico (%) | Precisión Específico (%) | Recall Específico (%) |
|---|---:|---:|---:|---:|---:|---:|
| Object Limits | 70.6% | 54.5% | 100.0% | 57.9% | 68.8% | 50.0% |
| Boxes Obscure Spheres | 71.2% | 55.3% | 100.0% | 0.0% | 0.0% | 0.0% |
| Stranger Shapes | 55.0% | 39.3% | 91.7% | 71.7% | 55.9% | 100.0% |
| Separated Boxes | 93.8% | 88.2% | 100.0% | 93.2% | 87.3% | 100.0% |
| Zzz | 76.5% | 65.0% | 92.9% | 16.7% | 100.0% | 9.1% |
| Match Silhouettes | 92.1% | 85.3% | 100.0% | 89.8% | 81.4% | 100.0% |
| Square Cross-Sections | 70.0% | 53.8% | 100.0% | 0.0% | 0.0% | 0.0% |
| Not Bird | 68.3% | 51.8% | 100.0% | 0.0% | 0.0% | 0.0% |
| More Than Meets Your Eye | 66.7% | 52.4% | 91.7% | 75.0% | 63.2% | 92.3% |
| Combine 2 Ramps | 89.6% | 81.2% | 100.0% | 85.7% | 75.0% | 100.0% |
| Warm Up | 92.6% | 86.2% | 100.0% | 91.5% | 84.4% | 100.0% |
| Pyramids are Strange | 84.2% | 72.7% | 100.0% | 78.4% | 64.4% | 100.0% |
| Stretch a Ramp | 85.0% | 73.8% | 100.0% | 90.3% | 82.3% | 100.0% |
| One Box | 97.7% | 95.5% | 100.0% | 95.8% | 92.0% | 100.0% |
| Sugar Cones | 93.3% | 87.5% | 100.0% | 88.1% | 78.8% | 100.0% |
| Removing Objects | 80.4% | 67.2% | 100.0% | 85.5% | 74.7% | 100.0% |
| Bull Market | 50.0% | 36.4% | 80.0% | 0.0% | 0.0% | 0.0% |
| Pi Henge | 91.9% | 85.0% | 100.0% | 92.3% | 85.7% | 100.0% |
| Tall and Small | 68.4% | 52.0% | 100.0% | 76.9% | 65.2% | 93.8% |
| 45-Degree Rotations | 91.1% | 83.7% | 100.0% | 79.5% | 66.0% | 100.0% |
| Angled Silhouette | 83.0% | 73.3% | 95.7% | 80.0% | 66.7% | 100.0% |
| Max 2 Boxes | 89.9% | 81.6% | 100.0% | 86.2% | 75.8% | 100.0% |
| Scaling Round Objects | 87.0% | 77.0% | 100.0% | 84.8% | 73.7% | 100.0% |
| Unnecessary | 75.0% | 60.0% | 100.0% | 75.0% | 60.0% | 100.0% |
| Bear Market | 7.4% | 4.0% | 50.0% | 0.0% | 0.0% | 0.0% |
| Rotate a Pyramid | 95.8% | 91.9% | 100.0% | 94.6% | 89.7% | 100.0% |
| Bird Fez | 81.7% | 70.7% | 96.7% | 77.4% | 63.2% | 100.0% |
| Few Clues | 40.0% | 25.0% | 100.0% | 0.0% | 0.0% | 0.0% |
| Orange Dance | 26.7% | 18.2% | 50.0% | 0.0% | 0.0% | 0.0% |
| Ramp Up and Can It | 77.8% | 63.6% | 100.0% | 72.7% | 63.2% | 85.7% |
| Tetromino | 100.0% | 100.0% | 100.0% | 80.0% | 66.7% | 100.0% |


En términos generales, el entrenamiento global ofrece un comportamiento más robusto, estable y consistente entre los distintos puzzles, manteniendo métricas equilibradas y una buena capacidad de generalización. Aunque el  específico puede mejorar el rendimiento en algunos casos concretos, también muestra una mayor variabilidad y sensibilidad al conjunto de datos utilizado. De hecho, en varios puzzles el modelo ajustado llega a obtener valores de precisión y F1 iguales a 0, lo que refleja fallos completos en la capacidad de predicción para determinadas tareas. En conjunto, estos resultados sugieren que el enfoque global proporciona una solución más fiable y menos propensa al sobreajuste, mientras que el entrenamiento especializado requiere condiciones más favorables y suficientes ejemplos representativos para resultar efectivo.

